# ⚽ Predict the FIFA World Cup 2026

## 📖 Background

The 2026 FIFA World Cup is one of the biggest sporting events in the world, hosted across the United States, Canada, and Mexico. For the first time, the tournament expands to 48 teams, producing 104 matches across the group stage and knockout rounds.

Using machine learning, historical statistics, and soccer domain knowledge, predict match scores, corners, and cards for every fixture. You must submit all your predictions before a single ball is kicked.

The scoring system rewards precision: an exact scoreline earns maximum points, while close predictions still earn partial credit. Later rounds carry score multipliers, so a strong model that holds up in the knockout stages can leapfrog the competition. The challenge is designed to be difficult enough that no one can achieve a perfect score—even with AI assistance—but accessible enough that any data enthusiast can participate and score points.


**Author:** Muhammad Zaki  
**Competition:** DataCamp FIFA World Cup 2026 Prediction Challenge  
**Deadline:** June 11, 2026 at 09:00 UTC  

---

## 🧠 Methodology

### Core Model — Dual-Poisson Distribution
The foundation of this notebook is a **Dual-Poisson statistical model**. Football goals are rare, random, and independent events, so the Poisson distribution is a natural fit. We estimate expected goals for both teams and simulate match outcomes to get the most likely scoreline and win/draw/loss probabilities.

Expected goals formulas:
- `λ_home = att_home × def_away × (1 + elo_factor) × (1 + home_advantage) × form × squad`
- `λ_away = att_away × def_home × (1 - elo_factor) × form × squad`

### What I Built
- Real World Cup 2018 + 2022 results as the statistical base, with WC 2022 weighted 2×
- Normalized attack and defense ratings so average team strength is centered at 1.0
- Team-level xG proxy integration — blends historical WC data (60%) with recent chance-quality estimates (40%)
- Dixon-Coles correction to reduce low-score bias, especially 0-0 overprediction
- Recent form factor from 2025/2026 performances
- Squad availability factor for injuries, suspensions, and aging-core concerns
- Improved red-card logic using a Poisson tension model
- Historical penalty shootout win rates for knockout tiebreakers
- Monte Carlo group-stage simulation (10,000 iterations)
- Best-third-place ranking for the 48-team format
- Increased simulation counts (1M) for Final and Semifinals
- Capped unrealistic scorelines for more credible outputs
- England adjusted downward based on poor recent form and squad concerns

### Data Sources
| Data | Source |
|------|--------|
| Competition fixtures | DataCamp (group_fixtures.csv, knockout_slots.csv) |
| WC 2018 + 2022 results | fixturedownload.com |
| Elo ratings | eloratings.net (May 2026) |
| Attack / defense base | fbref.com WC 2018+2022 averages |
| Corners per game | fbref.com WC 2018+2022 averages |
| Yellow / red card rates | fbref.com WC 2018+2022 averages |
| xG proxy values | Recent tournament and performance profiles |
| Recent form (2025/2026) | Nations League, WCQ results |
| Squad availability | Confirmed injuries and suspensions |
| Penalty records | WC + major tournament historical data |

### 🏆 Predicted Result
| Place | Team |
|-------|------|
| 🥇 Champion | Spain |
| 🥈 Runner-up | Argentina |
| 🥉 Third place | France |
| 4th place | Brazil |

*Spain beats Argentina in the Final on penalties at MetLife Stadium, New York.*

### Notebook Structure
| Cell | Description |
|------|-------------|
| Cell 2 | Imports and load competition CSV files |
| Cell 3 | Load real WC 2018+2022 results from fixturedownload.com |
| Cell 4 | Team stats database (Elo, corners, cards, host bonus) |
| Cell 5 | Override att/def with real data and normalize to avg=1.0 |
| Cell 6 | Recent form, squad availability, penalty record factors |
| Cell 7 | Team-level xG proxy table + blend into att/def |
| Cell 8 | Dixon-Coles correction function |
| Cell 9 | predict_match() — complete model function |
| Cell 10 | Backtest on WC 2022 — measure model accuracy |
| Cell 11 | Predict all 72 group stage matches |
| Cell 12 | Monte Carlo group simulation (10,000 iterations) |
| Cell 13 | Rank best 8 third-place teams |
| Cell 14 | Predict all 32 knockout matches |
| Cell 15 | Final summary and null check |

## 💾 The data

You have access to the following files:

#### `data/group_fixtures.csv` — all 72 group stage matches
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `group` | Group letter (A–L) |
| `home_team` | Home team name |
| `away_team` | Away team name |
| `date` | Match date (UTC) |
| `venue` | Stadium and city |

#### `data/knockout_slots.csv` — all 32 knockout round slots
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `round` | Round name (e.g. `Quarter-final`) |
| `multiplier` | Score multiplier for this round |
| `slot_home` | Description of the home team slot (e.g. `Winner Group A`) |
| `slot_away` | Description of the away team slot |

| Variable | Description |
|---|---|

You may also bring in any external data—FIFA rankings, historical match results, player statistics—to build your predictions.

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import urllib.request
import re, warnings
warnings.filterwarnings('ignore')

group_fixtures = pd.read_csv('data/group_fixtures.csv')
knockout_slots = pd.read_csv('data/knockout_slots.csv')

print(f"✅ Group fixtures : {len(group_fixtures)} matches")
print(f"✅ Knockout slots : {len(knockout_slots)} matches")
group_fixtures.head()

In [ ]:
knockout_slots = pd.read_csv('data/knockout_slots.csv')
knockout_slots

In [ ]:
# Load Real WC 2018+2022 Data
def fetch_wc_results(url, year):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as r:
        lines = r.read().decode('utf-8').strip().split('\n')
    rows = []
    for line in lines[1:]:
        parts = line.split(',')
        if len(parts) < 8: continue
        result = parts[7].strip().replace('"','')
        if ' - ' not in result: continue
        try:
            hs, as_ = result.split(' - ')
            rows.append({'home': parts[4].strip().replace('"',''),
                         'away': parts[5].strip().replace('"',''),
                         'home_score': int(hs), 'away_score': int(as_), 'year': year})
        except: continue
    return pd.DataFrame(rows)

wc18 = fetch_wc_results("https://fixturedownload.com/download/fifa-world-cup-2018-UTC.csv", 2018)
wc22 = fetch_wc_results("https://fixturedownload.com/download/fifa-world-cup-2022-UTC.csv", 2022)
historical = pd.concat([wc18, wc22], ignore_index=True)

NAME_MAP = {
    "United States": "USA", "Ivory Coast": "Côte d'Ivoire",
    "Cape Verde": "Cabo Verde", "Curacao": "Curaçao",
    "Korea Republic": "South Korea",
}
historical['home'] = historical['home'].replace(NAME_MAP)
historical['away'] = historical['away'].replace(NAME_MAP)
historical['weight'] = historical['year'].apply(lambda y: 2.0 if y == 2022 else 1.0)

print(f"✅ WC 2018 : {len(wc18)} matches")
print(f"✅ WC 2022 : {len(wc22)} matches")
print(f"✅ Total   : {len(historical)} matches (WC 2022 weighted 2×)")

In [ ]:
# Team Stats Database
# Format: (elo, att, def, corners, yellows, red_rate, host_bonus)
TEAM_STATS = {
    "Spain":           (2010, 1.90, 0.62, 5.8, 2.7, 0.04, 0.00),
    "Argentina":       (1985, 1.85, 0.68, 5.3, 2.9, 0.09, 0.00),
    "France":          (1970, 1.78, 0.72, 5.2, 3.0, 0.07, 0.00),
    "England":         (1880, 1.45, 0.92, 6.0, 2.6, 0.04, 0.00),
    "Brazil":          (1955, 1.80, 0.75, 5.5, 2.7, 0.06, 0.00),
    "Portugal":        (1945, 1.78, 0.78, 5.6, 2.8, 0.05, 0.00),
    "Netherlands":     (1940, 1.68, 0.78, 5.7, 2.9, 0.06, 0.00),
    "Germany":         (1930, 1.65, 0.80, 5.9, 2.7, 0.05, 0.00),
    "Belgium":         (1905, 1.58, 0.83, 5.4, 2.8, 0.06, 0.00),
    "USA":             (1855, 1.42, 0.98, 5.5, 2.8, 0.06, 0.20),
    "Mexico":          (1845, 1.38, 1.02, 5.2, 3.1, 0.09, 0.22),
    "Canada":          (1825, 1.32, 1.04, 5.0, 2.7, 0.05, 0.20),
    "Croatia":         (1845, 1.48, 0.88, 5.1, 3.0, 0.07, 0.00),
    "Morocco":         (1835, 1.32, 0.83, 5.0, 2.9, 0.08, 0.00),
    "Colombia":        (1825, 1.48, 0.98, 5.2, 3.2, 0.09, 0.00),
    "Uruguay":         (1815, 1.42, 0.93, 4.9, 3.3, 0.11, 0.00),
    "Switzerland":     (1805, 1.32, 0.88, 5.0, 2.6, 0.04, 0.00),
    "Japan":           (1795, 1.28, 0.93, 5.3, 2.5, 0.03, 0.00),
    "Senegal":         (1785, 1.28, 0.93, 4.8, 3.0, 0.08, 0.00),
    "South Korea":     (1775, 1.28, 0.98, 5.1, 2.7, 0.06, 0.00),
    "Ecuador":         (1765, 1.32, 1.03, 4.9, 3.1, 0.08, 0.00),
    "Iran":            (1765, 1.12, 1.03, 4.5, 3.2, 0.09, 0.00),
    "Austria":         (1755, 1.32, 0.98, 5.2, 2.8, 0.06, 0.00),
    "Australia":       (1735, 1.22, 1.03, 5.0, 2.8, 0.06, 0.00),
    "Norway":          (1725, 1.38, 0.98, 5.3, 2.7, 0.05, 0.00),
    "Côte d'Ivoire":   (1715, 1.28, 1.08, 5.0, 3.1, 0.09, 0.00),
    "Scotland":        (1705, 1.22, 1.08, 5.1, 2.9, 0.07, 0.00),
    "Ghana":           (1665, 1.22, 1.13, 4.7, 3.2, 0.09, 0.00),
    "Egypt":           (1685, 1.18, 1.03, 4.8, 3.0, 0.08, 0.00),
    "Algeria":         (1685, 1.22, 1.03, 4.9, 3.1, 0.08, 0.00),
    "Paraguay":        (1685, 1.18, 1.08, 4.7, 3.3, 0.11, 0.00),
    "Panama":          (1655, 1.08, 1.18, 4.5, 3.3, 0.11, 0.00),
    "Tunisia":         (1655, 1.08, 1.08, 4.6, 3.0, 0.08, 0.00),
    "Saudi Arabia":    (1645, 1.12, 1.13, 4.5, 3.1, 0.09, 0.00),
    "South Africa":    (1625, 1.08, 1.13, 4.4, 3.1, 0.09, 0.00),
    "Uzbekistan":      (1625, 1.12, 1.13, 4.4, 2.9, 0.07, 0.00),
    "Qatar":           (1615, 1.08, 1.18, 4.3, 3.0, 0.08, 0.00),
    "Cabo Verde":      (1605, 1.08, 1.13, 4.4, 3.2, 0.10, 0.00),
    "Jordan":          (1595, 1.02, 1.23, 4.2, 3.1, 0.09, 0.00),
    "New Zealand":     (1565, 0.92, 1.28, 4.2, 2.7, 0.06, 0.00),
    "Curaçao":         (1545, 0.92, 1.33, 4.0, 3.0, 0.09, 0.00),
    "Haiti":           (1515, 0.88, 1.38, 3.8, 3.1, 0.10, 0.00),
    "UEFA Playoff A":  (1770, 1.28, 1.00, 5.0, 2.9, 0.07, 0.00),
    "UEFA Playoff B":  (1760, 1.30, 0.99, 5.1, 2.9, 0.07, 0.00),
    "UEFA Playoff C":  (1750, 1.25, 1.02, 4.8, 2.9, 0.07, 0.00),
    "UEFA Playoff D":  (1740, 1.22, 1.04, 4.7, 2.9, 0.07, 0.00),
    "FIFA Playoff 1":  (1580, 0.95, 1.25, 4.2, 3.0, 0.08, 0.00),
    "FIFA Playoff 2":  (1640, 1.10, 1.15, 4.4, 3.0, 0.08, 0.00),
}

print(f"✅ {len(TEAM_STATS)} teams in database")
missing = [t for t in pd.concat([group_fixtures['home_team'],
           group_fixtures['away_team']]).unique() if t not in TEAM_STATS]
print(f"Missing: {missing if missing else 'None — all covered!'}")

In [ ]:
# Compute Real Stats from WC 2018+2022 Results
def compute_real_stats(team, df):
    hg = df[df['home'] == team]
    ag = df[df['away'] == team]
    if len(hg) + len(ag) < 2:
        return None
    scored, conceded, w = [], [], []
    for _, r in hg.iterrows():
        scored.append(r['home_score']); conceded.append(r['away_score']); w.append(r['weight'])
    for _, r in ag.iterrows():
        scored.append(r['away_score']); conceded.append(r['home_score']); w.append(r['weight'])
    return round(np.average(scored, weights=w), 3), round(np.average(conceded, weights=w), 3)

updated = 0
for team, vals in TEAM_STATS.items():
    result = compute_real_stats(team, historical)
    if result:
        att, defn = result
        TEAM_STATS[team] = (vals[0], att, defn, vals[3], vals[4], vals[5], vals[6])
        updated += 1
print(f"✅ {updated} teams updated with real WC data")

all_att = [v[1] for v in TEAM_STATS.values()]
all_def = [v[2] for v in TEAM_STATS.values()]
avg_att, avg_def = np.mean(all_att), np.mean(all_def)

for team, vals in TEAM_STATS.items():
    TEAM_STATS[team] = (
        vals[0],
        round(vals[1] / avg_att, 3),
        round(vals[2] / avg_def, 3),
        vals[3], vals[4], vals[5], vals[6]
    )
print(f"✅ Normalized — avg att: {avg_att:.3f}→1.000  avg def: {avg_def:.3f}→1.000")
print("ℹ️  xG blending will be applied in Cell 7.")

In [ ]:
# Additional Factors Form, Squad, Penalties
RECENT_FORM = {
    "Spain": 1.08, "Argentina": 1.06, "France": 1.05,
    "Morocco": 1.05, "USA": 1.05, "Germany": 1.03,
    "Netherlands": 1.02, "Japan": 1.03, "Colombia": 1.02,
    "Portugal": 0.97, "Belgium": 0.91, "England": 0.88,
    "Brazil": 0.95, "Mexico": 0.96, "Uruguay": 0.98,
    "Croatia": 0.95,
}

SQUAD_FACTOR = {
    "Belgium": 0.91, "England": 0.92, "Brazil": 0.93,
    "Portugal": 0.95, "France": 0.96, "Uruguay": 0.96,
    "Croatia": 0.94,
}

PENALTY_RECORD = {
    "Argentina": 0.78, "Germany": 0.75, "Croatia": 0.72,
    "Portugal": 0.63, "Spain": 0.60, "France": 0.57,
    "Brazil": 0.56, "Netherlands": 0.50, "Switzerland": 0.52,
    "Morocco": 0.55, "USA": 0.48, "Mexico": 0.45,
    "Colombia": 0.50, "Uruguay": 0.52, "Japan": 0.45,
    "Senegal": 0.48, "England": 0.42, "South Korea": 0.44,
}

print(f"✅ Recent form  : {len(RECENT_FORM)} teams")
print(f"✅ Squad factor : {len(SQUAD_FACTOR)} teams")
print(f"✅ Pen records  : {len(PENALTY_RECORD)} teams")

In [ ]:
# Team-level xG proxy table
# xg_for     = avg expected goals scored per game
# xg_against = avg expected goals conceded per game
XG_STATS = {
    "Spain":           {"xg_for": 2.10, "xg_against": 0.71},
    "France":          {"xg_for": 1.85, "xg_against": 0.82},
    "Argentina":       {"xg_for": 1.78, "xg_against": 0.88},
    "Brazil":          {"xg_for": 1.72, "xg_against": 0.85},
    "Portugal":        {"xg_for": 1.80, "xg_against": 0.90},
    "England":         {"xg_for": 1.35, "xg_against": 1.05},
    "Germany":         {"xg_for": 1.68, "xg_against": 0.95},
    "Netherlands":     {"xg_for": 1.62, "xg_against": 0.92},
    "Belgium":         {"xg_for": 1.48, "xg_against": 1.02},
    "USA":             {"xg_for": 1.35, "xg_against": 1.08},
    "Mexico":          {"xg_for": 1.30, "xg_against": 1.10},
    "Canada":          {"xg_for": 1.22, "xg_against": 1.12},
    "Croatia":         {"xg_for": 1.45, "xg_against": 0.95},
    "Morocco":         {"xg_for": 1.15, "xg_against": 0.82},
    "Colombia":        {"xg_for": 1.52, "xg_against": 1.05},
    "Uruguay":         {"xg_for": 1.38, "xg_against": 0.98},
    "Switzerland":     {"xg_for": 1.28, "xg_against": 0.92},
    "Japan":           {"xg_for": 1.22, "xg_against": 0.95},
    "Senegal":         {"xg_for": 1.18, "xg_against": 1.02},
    "South Korea":     {"xg_for": 1.15, "xg_against": 1.05},
    "Ecuador":         {"xg_for": 1.25, "xg_against": 1.08},
    "Austria":         {"xg_for": 1.35, "xg_against": 1.02},
    "Norway":          {"xg_for": 1.45, "xg_against": 1.05},
    "Australia":       {"xg_for": 1.10, "xg_against": 1.12},
    "Côte d'Ivoire":   {"xg_for": 1.18, "xg_against": 1.10},
    "Ghana":           {"xg_for": 1.08, "xg_against": 1.18},
    "Egypt":           {"xg_for": 1.05, "xg_against": 1.05},
    "Algeria":         {"xg_for": 1.12, "xg_against": 1.08},
    "Saudi Arabia":    {"xg_for": 1.02, "xg_against": 1.15},
    "Iran":            {"xg_for": 0.98, "xg_against": 1.08},
    "Tunisia":         {"xg_for": 0.95, "xg_against": 1.10},
    "Scotland":        {"xg_for": 1.10, "xg_against": 1.12},
    "Paraguay":        {"xg_for": 1.05, "xg_against": 1.15},
    "Uzbekistan":      {"xg_for": 1.08, "xg_against": 1.12},
    "South Africa":    {"xg_for": 0.98, "xg_against": 1.18},
    "Qatar":           {"xg_for": 0.92, "xg_against": 1.22},
    "Cabo Verde":      {"xg_for": 0.95, "xg_against": 1.15},
    "Jordan":          {"xg_for": 0.88, "xg_against": 1.25},
    "New Zealand":     {"xg_for": 0.85, "xg_against": 1.28},
    "Curaçao":         {"xg_for": 0.82, "xg_against": 1.35},
    "Haiti":           {"xg_for": 0.78, "xg_against": 1.42},
    "Panama":          {"xg_for": 0.92, "xg_against": 1.18},
}

print(f"✅ xG proxy data loaded — {len(XG_STATS)} teams")

# Blend: 60% historical WC + 40% xG proxy
XG_WEIGHT = 0.40
all_xg_for     = [v["xg_for"]     for v in XG_STATS.values()]
all_xg_against = [v["xg_against"] for v in XG_STATS.values()]
avg_xg_for     = np.mean(all_xg_for)
avg_xg_against = np.mean(all_xg_against)

blended = 0
for team, vals in TEAM_STATS.items():
    if team in XG_STATS:
        xg = XG_STATS[team]
        norm_xg_for     = xg["xg_for"]     / avg_xg_for
        norm_xg_against = xg["xg_against"] / avg_xg_against
        new_att = round((1 - XG_WEIGHT) * vals[1] + XG_WEIGHT * norm_xg_for,     3)
        new_def = round((1 - XG_WEIGHT) * vals[2] + XG_WEIGHT * norm_xg_against, 3)
        TEAM_STATS[team] = (vals[0], new_att, new_def, vals[3], vals[4], vals[5], vals[6])
        blended += 1

print(f"✅ xG blended into {blended} teams (60% historical WC + 40% xG proxy)")
print()
print(f"{'Team':<22} {'att (final)':>12} {'def (final)':>12}  {'xG dominance':>12}")
print("-" * 60)
for t in ["Spain","Argentina","England","Morocco","Belgium","Brazil","Japan","USA","Germany","France"]:
    if t in TEAM_STATS and t in XG_STATS:
        v = TEAM_STATS[t]
        dom = round(XG_STATS[t]["xg_for"] / XG_STATS[t]["xg_against"], 2)
        print(f"{t:<22} {v[1]:>12.3f} {v[2]:>12.3f}  {dom:>12.2f}x")

In [ ]:
# Dixon-Coles correction for low-scoring matches
RHO = -0.13

def dixon_coles_tau(hg, ag, lam_h, lam_a, rho=RHO):
    if   hg == 0 and ag == 0: return 1 - lam_h * lam_a * rho
    elif hg == 1 and ag == 0: return 1 + lam_a * rho
    elif hg == 0 and ag == 1: return 1 + lam_h * rho
    elif hg == 1 and ag == 1: return 1 - rho
    else:                     return 1.0

print("✅ Dixon-Coles correction ready (ρ = -0.13)")

In [ ]:
# Match Prediction Function with Caching
_cache = {}

def predict_match(home, away, neutral=False, n_sim=100_000):
    key = (home, away, neutral)
    if key in _cache:
        return _cache[key]

    def get(t):
        return TEAM_STATS.get(t, (1650, 1.00, 1.00, 4.5, 2.9, 0.08, 0.00))

    h, a = get(home), get(away)

    ef      = (h[0] - a[0]) / 3500
    ha      = (0.0 if neutral else 0.10) + h[6]
    form_h  = RECENT_FORM.get(home, 1.0)
    form_a  = RECENT_FORM.get(away, 1.0)
    squad_h = SQUAD_FACTOR.get(home, 1.0)
    squad_a = SQUAD_FACTOR.get(away, 1.0)

    lam_h = max(0.25, h[1] * a[2] * (1 + ef) * (1 + ha) * form_h * squad_h)
    lam_a = max(0.25, a[1] * h[2] * (1 - ef) * form_a * squad_a)

    rng = np.random.default_rng(abs(hash(str(key))) % (2**31))
    hg  = rng.poisson(lam_h, n_sim)
    ag  = rng.poisson(lam_a, n_sim)

    dc_weights = np.array([
        dixon_coles_tau(int(hg[i]), int(ag[i]), lam_h, lam_a)
        for i in range(n_sim)
    ])
    dc_weights = np.clip(dc_weights, 0.01, None)

    score_counts = {}
    for hg_, ag_, w in zip(hg, ag, dc_weights):
        k = (int(hg_), int(ag_))
        score_counts[k] = score_counts.get(k, 0) + w

    best = max(score_counts, key=score_counts.get)
    best = (min(best[0], 5), min(best[1], 4))

    total_w = dc_weights.sum()
    hw = float(np.sum(dc_weights[hg > ag])) / total_w
    dr = float(np.sum(dc_weights[hg == ag])) / total_w
    aw = float(np.sum(dc_weights[hg < ag])) / total_w

    oc = "home" if hw >= aw and hw >= dr else ("draw" if dr >= aw else "away")
    tc = h[3] + a[3]

    tension    = 1 + (h[5] + a[5]) * 2.5
    p_home_red = 1 - np.exp(-h[5] * tension)
    p_away_red = 1 - np.exp(-a[5] * tension)

    res = {
        "home_score":   best[0],  "away_score":   best[1],
        "outcome":      oc,
        "home_corners": round(tc * 0.55),
        "away_corners": round(tc * 0.45),
        "home_yellow":  max(1, round((h[4] + a[4]) / 2 * 0.9)),
        "away_yellow":  max(1, round((h[4] + a[4]) / 2 * 1.1)),
        "home_red":     1 if p_home_red > 0.12 else 0,
        "away_red":     1 if p_away_red > 0.12 else 0,
        "hw": hw, "dr": dr, "aw": aw,
        "pen_prob": round(min(0.35, dr + 0.08), 4),
    }
    _cache[key] = res
    return res

# Validation spot-check
print(f"{'Match':<35} {'Score':>5}  {'Out':>6}  {'H%':>5} {'D%':>5} {'A%':>5}  YC   RC")
print("-" * 78)
for h, a in [("Spain","Cabo Verde"), ("Argentina","Algeria"),
             ("England","Croatia"), ("Brazil","Morocco"),
             ("Belgium","Egypt"), ("USA","Paraguay")]:
    r = predict_match(h, a)
    print(f"{h+' vs '+a:<35} {r['home_score']}-{r['away_score']}  "
          f"{r['outcome']:>6}  {r['hw']*100:>5.1f} {r['dr']*100:>5.1f} "
          f"{r['aw']*100:>5.1f}  {r['home_yellow']}/{r['away_yellow']}  "
          f"{r['home_red']}/{r['away_red']}")

In [ ]:
# Backtest on WC 2022 Results
correct_outcome, correct_score, total = 0, 0, 0

for _, row in wc22.iterrows():
    home = NAME_MAP.get(row['home'], row['home'])
    away = NAME_MAP.get(row['away'], row['away'])
    if home not in TEAM_STATS or away not in TEAM_STATS:
        continue
    p = predict_match(home, away, neutral=False)
    real_oc = ("home" if row['home_score'] > row['away_score']
               else ("draw" if row['home_score'] == row['away_score'] else "away"))
    if p["outcome"] == real_oc:
        correct_outcome += 1
    if p["home_score"] == row['home_score'] and p["away_score"] == row['away_score']:
        correct_score += 1
    total += 1

print(f"WC 2022 Backtest ({total} matches):")
print(f"  Outcome accuracy    : {correct_outcome}/{total} = {correct_outcome/total*100:.1f}%")
print(f"  Exact score accuracy: {correct_score}/{total}  = {correct_score/total*100:.1f}%")

In [ ]:
# Generate Predictions for All Group Stage Matches
group_preds = []

for _, row in group_fixtures.iterrows():
    home, away = row["home_team"], row["away_team"]
    p = predict_match(home, away, neutral=False)
    group_preds.append({
        "match_id":     row["match_id"],
        "group":        row["group"],
        "home_team":    home,
        "away_team":    away,
        "home_score":   p["home_score"],
        "away_score":   p["away_score"],
        "outcome":      p["outcome"],
        "home_corners": p["home_corners"],
        "away_corners": p["away_corners"],
        "home_yellow":  p["home_yellow"],
        "away_yellow":  p["away_yellow"],
        "home_red":     p["home_red"],
        "away_red":     p["away_red"],
    })

group_preds_df = pd.DataFrame(group_preds)
print(f"✅ {len(group_preds_df)} group stage predictions complete")
print(f"   H: {(group_preds_df.outcome=='home').sum()}  "
      f"D: {(group_preds_df.outcome=='draw').sum()}  "
      f"A: {(group_preds_df.outcome=='away').sum()}  "
      f"Reds: {group_preds_df['home_red'].sum()+group_preds_df['away_red'].sum()}")
#group_preds_df

In [ ]:
# Simulate Group Stage Outcomes with Monte Carlo
GROUPS = {
    "A": ["Mexico","South Korea","South Africa","UEFA Playoff D"],
    "B": ["Canada","Qatar","Switzerland","UEFA Playoff A"],
    "C": ["Brazil","Morocco","Haiti","Scotland"],
    "D": ["USA","Paraguay","Australia","UEFA Playoff C"],
    "E": ["Germany","Côte d'Ivoire","Ecuador","Curaçao"],
    "F": ["Netherlands","Japan","UEFA Playoff B","Tunisia"],
    "G": ["Belgium","Egypt","Iran","New Zealand"],
    "H": ["Spain","Cabo Verde","Saudi Arabia","Uruguay"],
    "I": ["France","Senegal","FIFA Playoff 2","Norway"],
    "J": ["Argentina","Algeria","Austria","Jordan"],
    "K": ["Portugal","FIFA Playoff 1","Uzbekistan","Colombia"],
    "L": ["England","Croatia","Ghana","Panama"],
}

def simulate_group(teams, n_sim=10_000):
    fixtures = [(teams[i], teams[j]) for i in range(4) for j in range(i+1, 4)]
    rc = {t: {1:0, 2:0, 3:0, 4:0} for t in teams}
    pl = {t: [] for t in teams}
    gl = {t: [] for t in teams}
    fl = {t: [] for t in teams}
    rng = np.random.default_rng(2026)

    for _ in range(n_sim):
        pts = {t: 0 for t in teams}
        gd  = {t: 0 for t in teams}
        gf  = {t: 0 for t in teams}

        for home, away in fixtures:
            p = predict_match(home, away, neutral=False)
            r = rng.random()
            if r < p["hw"]:
                pts[home] += 3
                s = (max(1, p["home_score"]), p["away_score"])
            elif r < p["hw"] + p["dr"]:
                pts[home] += 1; pts[away] += 1
                s = (1, 1)
            else:
                pts[away] += 3
                s = (p["home_score"], max(1, p["away_score"]))
            gf[home] += s[0]; gf[away] += s[1]
            gd[home] += s[0]-s[1]; gd[away] += s[1]-s[0]

        ranked = sorted(teams, key=lambda t: (pts[t], gd[t], gf[t]), reverse=True)
        for i, t in enumerate(ranked):
            rc[t][i+1] += 1
        for t in teams:
            pl[t].append(pts[t]); gl[t].append(gd[t]); fl[t].append(gf[t])

    return {
        t: {
            "likely_rank": max(rc[t], key=rc[t].get),
            "p1st":    round(rc[t][1] / n_sim, 3),
            "p2nd":    round(rc[t][2] / n_sim, 3),
            "avg_pts": round(np.mean(pl[t]), 2),
            "avg_gd":  round(np.mean(gl[t]), 2),
            "avg_gf":  round(np.mean(fl[t]), 2),
        }
        for t in teams
    }

print("Running Monte Carlo simulation (10,000 iterations per group)...")
group_results  = {}
group_standings = {}

for grp, teams in GROUPS.items():
    res = simulate_group(teams)
    group_results[grp]   = res
    group_standings[grp] = sorted(teams, key=lambda t: res[t]["likely_rank"])
    g = group_standings[grp]
    print(f"  Group {grp}: 🥇 {g[0]:<22} 🥈 {g[1]:<20} 3rd: {g[2]}")

print("\n✅ Group simulation complete!")

In [ ]:
# Analyze Third-Place Teams Across All Groups
thirds = []
for grp, standings in group_standings.items():
    t = standings[2]
    s = group_results[grp][t]
    thirds.append({
        "group":   grp,
        "team":    t,
        "avg_pts": s["avg_pts"],
        "avg_gd":  s["avg_gd"],
        "avg_gf":  s["avg_gf"],
        "elo":     TEAM_STATS.get(t, (1650,))[0],
    })

thirds_df = pd.DataFrame(thirds).sort_values(
    ["avg_pts", "avg_gd", "avg_gf", "elo"],
    ascending=False
).reset_index(drop=True)

best8_thirds = thirds_df.head(8)["team"].tolist()

print("All 12 third-place teams ranked:")
print(thirds_df.to_string(index=False))
print(f"\n✅ Best 8 advancing third-place teams:")
for i, t in enumerate(best8_thirds, 1):
    print(f"  {i}. {t}")

In [ ]:
# Knockout Stage Predictions
used_thirds  = []
match_winner = {}
match_loser  = {}
ko_preds     = []

def resolve_ko_slot(slot_str):
    s = str(slot_str).strip()
    for g in "ABCDEFGHIJKL":
        if s == f"Winner Group {g}":    return group_standings[g][0]
        if s == f"Runner-up Group {g}": return group_standings[g][1]
    if "Best 3rd" in s:
        m = re.search(r'\(Groups? ([A-L/]+)\)', s)
        letters = re.findall(r'[A-L]', m.group(1)) if m else []
        for g in letters:
            t = group_standings[g][2]
            if t not in used_thirds:
                used_thirds.append(t); return t
        for t in best8_thirds:
            if t not in used_thirds:
                used_thirds.append(t); return t
    return "TBD"

def penalty_winner(home, away):
    ph = PENALTY_RECORD.get(home, 0.50)
    pa = PENALTY_RECORD.get(away, 0.50)
    return home if (ph / (ph + pa)) >= 0.50 else away

for _, row in knockout_slots.iterrows():
    mid  = int(row["match_id"])
    rnd  = row["round"]
    mult = int(row["multiplier"])

    def resolve(s):
        s = str(s).strip()
        if s.startswith("Winner Match "): return match_winner.get(int(s.split()[-1]), "TBD")
        if s.startswith("Loser Match "):  return match_loser.get(int(s.split()[-1]), "TBD")
        return resolve_ko_slot(s)

    home = resolve(row["slot_home"])
    away = resolve(row["slot_away"])

    sims = {16: 1_000_000, 8: 500_000}.get(mult, 100_000)
    p = predict_match(home, away, neutral=True, n_sim=sims)

    goes_to_pens = (p["pen_prob"] > 0.30) and (abs(p["hw"] - p["aw"]) < 0.12)
    winner = penalty_winner(home, away) if goes_to_pens else (home if p["hw"] >= p["aw"] else away)
    loser  = away if winner == home else home

    match_winner[mid] = winner
    match_loser[mid]  = loser

    ko_preds.append({
        "match_id":          mid,
        "round":             rnd,
        "multiplier":        mult,
        "home_team":         home,
        "away_team":         away,
        "home_score":        p["home_score"],
        "away_score":        p["away_score"],
        "outcome":           "home" if winner == home else "away",
        "goes_to_penalties": goes_to_pens,
        "home_corners":      p["home_corners"],
        "away_corners":      p["away_corners"],
        "home_yellow":       p["home_yellow"],
        "away_yellow":       p["away_yellow"],
        "home_red":          p["home_red"],
        "away_red":          p["away_red"],
    })

ko_preds_df = pd.DataFrame(ko_preds)
print(f"✅ {len(ko_preds_df)} knockout predictions complete")
print()
for _, r in ko_preds_df.iterrows():
    w   = r['home_team'] if r['outcome'] == 'home' else r['away_team']
    pen = " [PENS]" if r['goes_to_penalties'] else ""
    ico = "🏆" if r['round'] == 'Final' else ("🥉" if 'Third' in r['round'] else "→")
    print(f"M{r['match_id']:3d} [{r['round'][:9]:9s} ×{r['multiplier']:2d}]  "
          f"{r['home_team']:22s} vs {r['away_team']:22s}  "
          f"{r['home_score']}-{r['away_score']}  {ico} {w}{pen}")

In [ ]:
# Final Summary and Checks
print("=" * 65)
print("GROUP STAGE PREDICTIONS — 72 matches")
print("=" * 65)
display(group_preds_df)

print("\n" + "=" * 65)
print("KNOCKOUT PREDICTIONS — 32 matches")
print("=" * 65)
display(ko_preds_df)

print("\n" + "=" * 65)
print("TOURNAMENT FINAL RESULT")
print("=" * 65)
print(f"  🏆 Champion    : {match_winner.get(104, 'TBD')}")
print(f"  🥈 Runner-up   : {match_loser.get(104, 'TBD')}")
print(f"  🥉 Third place : {match_winner.get(103, 'TBD')}")
print(f"  4th place      : {match_loser.get(103, 'TBD')}")
print(f"\n  Group stage reds   : {group_preds_df['home_red'].sum() + group_preds_df['away_red'].sum()}")
print(f"  Knockout penalties : {ko_preds_df['goes_to_penalties'].sum()}")

print("\n" + "=" * 65)
print("✅ NULL / TBD CHECK")
print("=" * 65)
gp_nulls = group_preds_df.isnull().sum().sum()
ko_nulls = ko_preds_df.isnull().sum().sum()
tbd_home = (ko_preds_df['home_team'] == 'TBD').sum()
tbd_away = (ko_preds_df['away_team'] == 'TBD').sum()
print(f"  Group pred nulls : {gp_nulls}")
print(f"  Knockout nulls   : {ko_nulls}")
print(f"  TBD home teams   : {tbd_home}")
print(f"  TBD away teams   : {tbd_away}")

if gp_nulls == 0 and ko_nulls == 0 and tbd_home == 0 and tbd_away == 0:
    print("\n✅ ALL CLEAR — Safe to publish! 🚀")
else:
    print("\n❌ ISSUES FOUND — Fix before publishing!")